<div style="text-align:right;color:blue">version id: __VERSION_ID__</div>

# Runtime of different Fibonacci implementations
In the tickables we implemented several functions for computing the $n$-th Fibonacci number $F_n$. We also discovered that for large $n$ the runtime of these implementations differed significantly. In this notebook we will analyse the performance of those different implementations more systematically, and we will explain why for some implementations the runtime increases dramatically for large values of $n$. For this, we will use the order of growth analysis and big-O notation that we developed last week.

<hr style="height: 2px">

### What you will learn
In this notebook we will cover the following topics:

* Review of different Fibonacci implementations
* Measured runtimes
* Analysis of iterative and recursive implementations
* Performance of matrix-based implementations
* Comparison of computational complexities

<hr style="height: 2px">

*&#169; Eike Mueller, University of Bath 2019-2021. These lecture notes are copyright of Eike Mueller, University of Bath. They are provided exclusively for educational purposes at the University and are to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## Review of Fibonacci implementations

Recall that the $n$-th Fibonacci number $F_n$ is defined as follows:

$$
F_0 = 0,\qquad F_1 = 1,\qquad F_n = F_{n-1} + F_{n-2} \quad\text{for $n\ge 2$.}
$$

We implemented several functions for computing $F_n$:

1. **Iterative:** The function `fib_iter(n)` computes $F_n$ iterately using a for-loop.
2. **Recursive:** In `fib_rec(n)` the $n$-th Fibonacci number is computed by recusively calling the same function with $n-1$ and $n-2$.
3. **Matrix-based:** We wrote two implementations which use the fact that

$$
\begin{pmatrix}
F_n\\F_{n-1}
\end{pmatrix}
=
\begin{pmatrix}
1&1\\
1&0
\end{pmatrix}^n
\begin{pmatrix}
0\\1
\end{pmatrix}
$$

* First, we computed $F_n$ by repeatedly multiplying the vector $\begin{pmatrix}0\\1\end{pmatrix}$ with the matrix $A_{\text{fib}}=\begin{pmatrix}1&1\\1&0\end{pmatrix}$; this was realised through a recursive call in the function `fib_rec_mat(n)`.
* Next, we optimised this by raising the matrix $A_{\text{fib}}$ to the $n$-th power with the `fast_power_mat()` function in `fib_rec_fast_mat(n)`

For large values of $n$ the runtime differs significantly. We can check this by using the `%timeit` statement. The following cell computes $F_n$ with $n=15$. Execute it and see how the runtime changes for different values of $n$. If a calculation for a particular value of $n$ takes too long, you can always interupt it by clicking on ''Kernel'' $\rightarrow$ ''Interrupt'' from the menu bar above.

In [1]:
from model_solutions import fib_iter, fib_rec, fib_rec_mat, fib_rec_fast_mat

n = 15

print ('*** iterative implementation fib_iter() ***')
%timeit fib_iter(n)
print ('*** recursive implementation fib_rec() ***')
%timeit fib_rec(n)
print ('*** matrix based implementation fib_rec_mat() ***')
%timeit fib_rec_mat(n)
print ('*** matrix based implementation with fast_power_mat(), fib_rec_mat() ***')
%timeit fib_rec_fast_mat(n)

*** iterative implementation fib_iter() ***
1.11 µs ± 57.8 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)
*** recursive implementation fib_rec() ***
450 µs ± 4.73 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
*** matrix based implementation fib_rec_mat() ***
57.2 µs ± 1.64 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)
*** matrix based implementation with fast_power_mat(), fib_rec_mat() ***
13.2 µs ± 104 ns per loop (mean ± std. dev. of 7 runs, 100000 loops each)


Converting everything to microseconds ($1\text{microsecond} = 1\mu s = 10^{-6}s$) I get the following results on my computer:

| method | runtime \[microseconds \] |
| --- | --- |
| `fib_iter()` | 0.63 |
| `fib_rec()` | 223 |
| `fib_rec_mat()` | 35.5 |
| `fib_rec_fast_mat()` | 9.3 |

This table confirms that there is a vast different between the methods. `fib_iter()` is fastest overall and `fib_rec()` is more than $200\times$ slower! Comparing the two matrix based methods, using `fast_power_mat()` reduces the runtime significantly.

## Measured runtimes
As always, we are particularly interested in what happens as $n$ becomes very large, i.e. we want to know how the runtime grows as $n$ increases.

The following plot shows the measured runtimes for a range of $n$.

![Runtimes of Fibonacci implementations](./figures/fibonacci_runtime_total.png "Runtimes of fib_iter(), fib_rec() and fib_rec_mat()")

One striking feature is that the time it takes to execute `fib_rec()` explodes as $n$ becomes large. `fib_iter()` and `fib_rec_mat()` are much faster, and the runtime grows much less rapidly for large $n$. To have a closer look at those times, the following plot shows the same data, but the measured times for the `fib_iter()` implementation is multiplied by a factor 100:

![Rescaled runtimes of Fibonacci implementations](./figures/fibonacci_runtime_total_rescaled.png "Runtimes of fib_iter(), fib_rec() and fib_rec_mat(), rescaled to simplify comparisons")

As this plot reveals, the time $T(n)$ it takes to run `fib_iter()` and `fib_rec_mat()` to compute $F_n$ can be described very well by a function of the form $\alpha + \beta n$ (where the constants $\alpha$ and $\beta$ depend on the implementation and the computer that runs the code). Since $\alpha + \beta n=\Theta(n)$, this implies that empirically $T(n)=\Theta(n)$. In other words, for very large values of $n$ the runtime increases by a factor of approximately two if $n$ is doubled.

## Analysis of iterative and recursive implementations
Having explored the growth in runtime empirically with our numerical experiments, we now analyse it theoretically by counting the elops in the different implementations.
### Runtime of `fib_iter()`
To understand the growth in runtime for `fib_iter()` let us have a look at the algorithm in [Tickable 3](../../../tickables/S1/Tickable3.ipynb), which is written down here for the case that $n\ge 2$:

### Algorithm: Calculate $n$-th Fibonacci number $F_n$ for $n\ge 2$ with an iterative method
**Input** $n\ge 2$
**Output** $n$-th Fibonacci number
1. Initialise $\alpha\mapsto 0$; $\beta\mapsto 1$
1. **for** $k=1,\dots,n-1$
1. $~~~~$ $\gamma \mapsto \beta+\alpha$
1. $~~~~$ $\alpha \mapsto \beta$
1. $~~~~$ $\beta \mapsto \gamma$
1. **end for**
1. **return** $\gamma$

To count the number of elops, note that only one addition is carried out in line 3 in the loop body. Since we loop from $k=1$ to $n-1$, the loop body is executed $n-1$ times, and hence the number of elops is

$$
E_{\text{fib_iter}}(n) = n-1 = \Theta(n)
$$

Since the runtime is proportional to the number of elops, we therefore expect that

$$
T_{\text{fib_iter}}(n)=(n-1)\tau = \Theta(n)
$$

where $\tau$ is the time it takes to carry out one elop. This agrees with our numerical experiments for large values of $n$.

There is a small subtlety here: looking at the full code in [Tickable 3](../../../tickables/S1/Tickable3.ipynb) we also note that the number of elops is zero for $n=0$ and $n=1$. So by only counting the number of elops, we might conclude that the runtime is zero for $n=0$ and $n=1$. Looking at the curve in the figure above this does not appear to be the case. The reason for this is that we have to pay some fixed cost for calling the function, in other words, executing a function which does nothing in Python is not free. The following code demonstrates this:

In [2]:
def do_nothing():
    pass # the 'pass' command does nothing

%timeit do_nothing()

72.7 ns ± 1.68 ns per loop (mean ± std. dev. of 7 runs, 10000000 loops each)


The cost for ``doing nothing`` is very small, less than $100\text{ns}=10^{-7}\text{s}$. Furthermore, in our case this startup cost does not matter, since we only do this once. If the cost for calling a function  is $t_{\text{startup}}$, then taking this into account our model for the runtime would change to

$$
T_{\text{fib_iter}}(n)=(n-1)\tau + t_{\text{startup}} = \Theta(n)
$$

In other words, adding a constant does not change the $\Theta(n)$ growth in runtime, which is what we are interested in after all.

### Runtime of `fib_rec()`
Let us now look at `fib_rec()`. The recursive algorithm for computing $F_n$ is given by

### Function $\text{fib_rec(n)}$: Calculate $n$-th Fibonacci number $F_n$ recursively
**Input** $n\ge 2$
**Output** $n$-th Fibonacci number
1. **if** $n==0$ **then**
1. $~~~~$ **return** 0
1. **else if** $n==1$ **then**
1. $~~~~$ **return** 1
1. **else**
1. $~~~~$ **return** $\text{fib_rec(n-1)}+\text{fib_rec(n-2)}$
1. **end if**

How many elops $E_{\text{fib_rec}}(n)$ are needed for executing this algorithm for a given $n$? If $n\ge 2$, we have to take into account the cost of one addition, plus the cost for calling the function for $n-1$ and $n-2$. If $n=0$ or $n=1$ the number of elops is zero. This implies that the number of elops is given by the following recursion relation.

$$
E_{\text{fib_rec}}(n) = \begin{cases}
1 + E_{\text{fib_rec}}(n-1) + E_{\text{fib_rec}}(n-2) & \text{for $n\ge 2$}\\
0 & \text{for $n=0$ or $n=1$}
\end{cases}
$$

Let us now define $G_{n+1} = E_{\text{fib_rec}}(n)+1$ for all $n\ge 0$. Then we find that $G_1=G_2=1$ and $G_{n+1} = G_n + G_{n-1}$ for $n\ge 2$, or (replacing $n+1\rightarrow n$):

$$
G_n = G_{n-1}+ G_{n-2} \qquad \text{for $n\ge 3$}
$$

This, however, is the same recursion relation as for the Fibonacci numbers. Since also $F_1=F_2=1$, $G_n$ is the same as the $n$-th Fibonacci number $F_n$ and we find for the number of elops:

$$
E_{\text{fib_rec}}(n) = F_{n+1}-1
$$

Next, note that we can write

$$
F_n = \frac{1}{\sqrt{5}}\left(\left(\frac{1+\sqrt{5}}{2}\right)^n-\left(\frac{1-\sqrt{5}}{2}\right)^n\right)
\qquad(\dagger)
$$

(we will not prove this identity in this course).

Since $\frac{1}{\sqrt{5}}\left|\frac{1-\sqrt{5}}{2}\right|^n < 1$, Eq. ($\dagger$) implies that

$$
F_n \le \frac{1}{\sqrt{5}}\left(\frac{1+\sqrt{5}}{2}\right)^n + 1.
$$

and therefore

$$
E_{\text{fib_rec}}(n) \le \frac{1}{\sqrt{5}}\left(\frac{1+\sqrt{5}}{2}\right)^{n+1}
$$

from which we can deduce that

$$
E_{\text{fib_rec}}(n) = \mathcal{O}\left(\left(\frac{1+\sqrt{5}}{2}\right)^n\right)
$$

In fact, using Eq. ($\dagger$), one can also show that $F_n \ge \frac{1}{2\sqrt{5}}\left(\frac{1+\sqrt{5}}{2}\right)^n-1$ for $n\ge 5$, which implies that $E_{\text{fib_rec}}(n) = \Omega\left(\left(\frac{1+\sqrt{5}}{2}\right)^n\right)$ and therefore $E_{\text{fib_rec}}(n) = \Theta\left(\left(\frac{1+\sqrt{5}}{2}\right)^n\right)$. For very large values of $n$ the second term in Eq. ($\dagger$) becomes very small and we expect that in this case the runtime is given approximately by

$$
T_{\text{fib_rec}}(n)\approx C\cdot \rho^n\qquad\text{where $\rho = \frac{1+\sqrt{5}}{2}\approx 1.618$}
$$

and $C$ is a constant. This means that the runtime grows *exponentially*. If we increase $n$ by one, the runtime will grow by a factor $\rho\approx 1.618$. If we plot the logarithm of the runtime as a function of $n$, which is equivalent to using a logarithmic scale on the vertical axis of our plot, we would expect to see a straight line since

$$
\log\left(T_{\text{fib_rec}}(n)\right)\approx \log(C) + n \cdot\log{\rho}\qquad(\ddagger)
$$

The following plot shows the measured runtimes with a logarithmic scale on the vertical axis. The plot confirms that indeed Eq. ($\ddagger$) describes the measured times very well:

![Runtime of recursive Fibonacci implemetation](./figures/fibonacci_runtime_logarithmic_fib_rec.png "Runtimes of fib_rec() with logarithmic vertical axis")

## Performance of matrix-based implementations
The function `fib_rec_mat()` relies on the following recursion for computing the vector $\begin{pmatrix}F_n\\F_{n-1}\end{pmatrix}$:

### Function $\text{fib_rec_mat_vec}(n)$: Calculate $n$-th Fibonacci vector recursively
**Input** $n\ge 0$
**Output** $n$-th Fibonacci vector $\begin{pmatrix}F_n\\F_{n-1}\end{pmatrix}$
1. **if** $n==0$ **then**
1. $~~~~$ **return** $\begin{pmatrix}0\\1\end{pmatrix}$
1. **else**
1. $~~~~$ **return** $\begin{pmatrix}1 & 1\\1 & 0\end{pmatrix}\cdot \text{fib_rec_mat_vec}(n-1)$
1. **end if**

If $n\ge 1$ we multiply a two-dimensional vector (which is obtained with a recursive call) by a $2\times 2$ matrix, i.e. we compute a product of this form:

$$
\begin{pmatrix}
A_{00} & A_{01} \\
A_{10} & A_{11}
\end{pmatrix}
\cdot
\begin{pmatrix}
u_0 \\ u_1
\end{pmatrix}
=
\begin{pmatrix}
A_{00}\cdot u_0 + A_{01}\cdot u_1\\ A_{10}\cdot u_0 + A_{11}\cdot u_1
\end{pmatrix}
$$

This requires two additions and four multiplications, i.e. six elops in total. No computations are necessary if $n=0$. Hence we find that the number of elops for a given $n$ satisfies

$$
E_{\text{fib_rec_mat_vec}}(n) = \begin{cases}
6 + E_{\text{fib_rec_mat_vec}}(n-1) & \text{if $n \ge 1$}\\
0 & \text{if $n=0$}
\end{cases}
$$

This recusion relation is readily solved to obtain

$$
E_{\text{fib_rec_mat_vec}}(n) = 6n.
$$

We therefore expect the runtime of `fib_rec_mat()` to grow linearly with $n$, which is exactly what we observe. Per iteration it requires more elops that `fib_iter()`, which is why `fib_rec_mat()` is slower that `fib_iter()` for a given value of $n$.

Finally, consider `fib_rec_fast_mat()`, which uses the fast matrix power algorithm to compute $A^n$, the $n$-th power of a matrix $A$. The algorithm for raising a number $x$ to the $n$-th power was discussed in [this notebook](../week06/Lecture_06b_DivideAndConquer.ipynb), where it was found that the number $\nu(n)$ of recursive calls satisfies $\nu(n)\le \log_2(n)$. The total number of elops can then be bounded by the product of $\nu(n)$ and the maximal number of elops per call. If the algorithm is used to compute the matrix power $A^n$, we have to compute up to two $2\times 2$ matrix product of the form

$$
\begin{pmatrix}
A_{00} & A_{01}\\
A_{10} & A_{11}
\end{pmatrix}
\cdot
\begin{pmatrix}
B_{00} & B_{01}\\
B_{10} & B_{11}
\end{pmatrix}
=
\begin{pmatrix}
A_{00}\cdot B_{00} + A_{01}\cdot B_{10} & 
A_{00}\cdot B_{01} + A_{01}\cdot B_{11} \\
A_{10}\cdot B_{00} + A_{11}\cdot B_{10} & 
A_{10}\cdot B_{01} + A_{11}\cdot B_{11}
\end{pmatrix}
$$

each of which requires 12 elops. Hence the total number of elops can be bounded by

$$
E_{\text{fib_rec_fast_mat}}(n) \le 12 \nu(n) \le 12\log_2(n) = \mathcal{O}(\log_2(n))
$$

It is also possible to show that $E_{\text{fib_rec_fast_mat}}(n) = \Omega(\log_2(n))$, and hence $E_{\text{fib_rec_fast_mat}}(n) = \Theta(\log_2(n))$. We therefore expect the runtime to grow *logarithmically* for large values of $n$, and this is indeed what we observe.

The following figure shows the measured runtimes for the two matrix-based implementations:

![Runtimes of matrix-based Fibonacci implementations](./figures/fibonacci_runtime_matrix.png "Runtimes of fib_rec_mat() and fib_rec_fast_mat()")

## Comparison
For reference, the computational complexity of the different implementations is summarised in the following table:

| Method | Implementation | Complexity | big-O notation |
| --- | --- | --- | --- |
| Iterative | `fib_iter()` | linear | $\Theta(n)$ |
| Recursive | `fib_rec()` | exponential | $\Theta(\rho^n)$, $\rho \approx 1.618$ |
| Matrix based | `fib_rec_mat()` | linear | $\Theta(n)$ |
| Matrix based, fast power method | `fib_rec_fast_mat()` | logarithmic | $\Theta(\log_2(n))$ |

This concludes our discussion of the performance of different implementations of the Fibonacci numbers.

## Proof of alternative representation of Fibonacci numbers (not examinable)
In this section we will sketch out a proof for the identity

$$
F_n = \frac{1}{\sqrt{5}}\left(\left(\frac{1+\sqrt{5}}{2}\right)^n-\left(\frac{1-\sqrt{5}}{2}\right)^n\right)
$$

that has been written down in Eq. ($\dagger$) above. This proof requires some technical manipulations and it is not essential for this course. Nevertheless the main steps are written down here for completeness.

First, we need two statements about the matrix $A_{\text{fib}}=\begin{pmatrix}1&1\\1&0\end{pmatrix}$:

### Statement 1
Consider the $2\times 2$ matrix $U$ given by

$$
U = \begin{pmatrix}
\sqrt{\frac{\sqrt{5}+1}{2\sqrt{5}}} &
-\sqrt{\frac{\sqrt{5}-1}{2\sqrt{5}}} \\
\sqrt{\frac{\sqrt{5}-1}{2\sqrt{5}}} &
\sqrt{\frac{\sqrt{5}+1}{2\sqrt{5}}} \\
\end{pmatrix}.
$$

Then the product of the three matrices $U^T$, $A_{\text{fib}}$ and $U$ is a diagonal $2\times2$ matrix $D$ given by

$$
U^T A_{\text{fib}} U = D = \begin{pmatrix}\frac{\sqrt{5}+1}{2}&0\\0&\frac{\sqrt{5}-1}{2}\end{pmatrix}.
$$
### Statement 2
The product of the two matrices $U$ and $U^T$ is the $2\times2$ identity matrix $I_{2\times2}$:

$$
UU^T = I_{2\times 2} = \begin{pmatrix}1&0\\0&1\end{pmatrix}
$$

Both statements can be proved by mechanically applying the rules for matrix-matrix multiplication laid out in the [notebook on vectors and matrices](../week01/Lecture_01a_VectorsMatrices.ipynb). Admittedly the manipulations are quite tedious, but all that is required is some patience to follow the technical calculations. In the language of linear algebra, the two statements say that *''the orthogonal matrix $U$ diagonalises the matrix $A_{\text{fib}}$''*. Don't worry if this does not make much sense to you right now, it will be discussed in much more detail in your course on algebra.

Next, we use the fact that

$$
\begin{pmatrix}
F_n\\F_{n-1}
\end{pmatrix}
= \begin{pmatrix}
1 & 1\\
1 & 0
\end{pmatrix}^n
\begin{pmatrix}
0\\1
\end{pmatrix}
=
A_{\text{fib}}^n
\begin{pmatrix}
0\\1
\end{pmatrix}
$$

and rewrite the right hand side in a clever way as follows:

$$
\begin{aligned}
\begin{pmatrix}
F_n\\F_{n-1}
\end{pmatrix}
&= \underbrace{A_{\text{fib}} A_{\text{fib}} A_{\text{fib}} \dots  A_{\text{fib}}}_{\text{$n$}\; times}
\begin{pmatrix}
0\\1
\end{pmatrix}\\
&= I_{2\times 2} A_{\text{fib}} I_{2\times 2} A_{\text{fib}} I_{2\times 2} A_{\text{fib}} I_{2\times 2} \dots   I_{2\times 2}A_{\text{fib}}  I_{2\times 2}
\begin{pmatrix}
0\\1
\end{pmatrix}\\
&= UU^T A_{\text{fib}} UU^T A_{\text{fib}} UU^T A_{\text{fib}} UU^T \dots UU^TA_{\text{fib}}  UU^T
\begin{pmatrix}
0\\1
\end{pmatrix}\\
    &= U\underbrace{\left(U^T A_{\text{fib}} U\right)}_{=D}\underbrace{\left(U^T A_{\text{fib}} U\right)}_{=D}\underbrace{\left(U^T A_{\text{fib}} U\right)}_{=D}\left(U^T \dots U\right)\underbrace{\left(U^TA_{\text{fib}}  U\right)}_{=D}U^T
\begin{pmatrix}
0\\1
\end{pmatrix}\\
&=U D^{n} U^T
\begin{pmatrix}
0\\1
\end{pmatrix}\qquad(*)
\end{aligned}
$$

Since $D$ is a diagonal $2\times 2$ matrix, $D^n$ is also a diagonal $2\times 2$ matrix given by

$$
D^n = \begin{pmatrix}\left(\frac{\sqrt{5}+1}{2}\right)^n&0\\0&\left(\frac{\sqrt{5}-1}{2}\right)^n\end{pmatrix}.
$$

Now all that remains is to compute the matrix-matrix and matrix-vector products in the final line of Eq. $(*)$. In fact, to obtain $F_n$ it sufficies to compute the first component of the following vector:

$$
\begin{equation}
U D^{n} U^T
\begin{pmatrix}
0\\1
\end{pmatrix}
\end{equation}
$$

Although again the manipulations are tedious, by mechanically applying the rules for matrix-vector multiplication this results in the expression written down in Eq. ($\dagger$).